# Corner held-out far-to-near training-data sweep with retrained surrogate

This notebook trains the corner held-out far-to-near data-amount sweep and saves the tabular results plus every trained model.

For each of ten training fractions and the selected seed, it retrains the forward surrogate on the same limited training subset used for the inverse model. The surrogate architecture is copied from the current saved best surrogate model, and the training settings follow the surrogate training notebook (`ml_11`). The retrained surrogate is then frozen while the inverse model is trained against it.

It writes:

- `data_amount_sweep_corner_far_to_near_retrained_surrogate.csv`
- `data_amount_sweep_corner_far_to_near_retrained_surrogate_summary.csv`
- `model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_*_surrogate.keras`
- `model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_*_combined.keras`
- `model/corner_far_to_near_data_amount_sweep_retrained_surrogate/fraction_*_inverse.keras`

Plot generation lives in `ml_34_corner_far_to_near_sweep_plots.ipynb` so figures can be regenerated without retraining the models.

The important interpretation is that 100% means 100% of the non-held-out training pool. The validation and test corner samples are still excluded.


In [10]:
from __future__ import annotations

import gc
import json
import os
import sys
import zipfile
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.neighbors import NearestNeighbors
from tensorflow.keras import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LeakyReLU
from tensorflow.keras.models import load_model


tf.keras.backend.set_floatx("float32")


# This notebook can be run either from the transmon experiment folder or from the
# repo root. The block below tries both so local paths do not need hard-coding.
HERE = Path.cwd()
EXPERIMENT_RELATIVE = Path("experiments/model_predict_qubit_TransmonCross_Hamiltonian_params")

if (HERE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE
elif (HERE / EXPERIMENT_RELATIVE / "metadata" / "qubit-TransmonCross-Hamiltonian_params.json").exists():
    EXPERIMENT_DIR = HERE / EXPERIMENT_RELATIVE
else:
    raise FileNotFoundError(
        "Could not find the transmon-cross metadata file. "
        "Run this notebook from the repo root or from the transmon experiment folder."
    )

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

from parameters_surrogate_defined_loss import (  # noqa: E402
    EPOCHS,
    MODEL_DIR as PARAM_MODEL_DIR,
    SCALERS_DIR as PARAM_SCALERS_DIR,
    TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS,
)
from parameters_surrogate import (  # noqa: E402
    EPOCHS as SURROGATE_EPOCHS,
    TRAIN_BATCH_SIZE as SURROGATE_TRAIN_BATCH_SIZE,
    TRAIN_EARLY_STOPPING_PATIENCE as SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
    TRAIN_LOSS as SURROGATE_TRAIN_LOSS,
)

METADATA_DIR = EXPERIMENT_DIR / "metadata"
METADATA_PATH = METADATA_DIR / "qubit-TransmonCross-Hamiltonian_params.json"
OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_corner_far_to_near_retrained_surrogate.csv"
SUMMARY_OUT_PATH = EXPERIMENT_DIR / "data_amount_sweep_corner_far_to_near_retrained_surrogate_summary.csv"

MODEL_DIR = Path(PARAM_MODEL_DIR)
SCALERS_DIR = Path(PARAM_SCALERS_DIR)
SWEEP_MODEL_DIR = MODEL_DIR / "corner_far_to_near_data_amount_sweep_retrained_surrogate"
SWEEP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
SURROGATE_MODEL_PATH = MODEL_DIR / "best_keras_model_model2_surrogate.keras"
COMBINED_MODEL_CANDIDATES = [
    MODEL_DIR / "surrogate_loss_2in_3out_best_model.keras",
    MODEL_DIR / "best_keras_model_surrogate_defined_loss.keras",
]
COMBINED_MODEL_PATH = next((path for path in COMBINED_MODEL_CANDIDATES if path.exists()), None)
if COMBINED_MODEL_PATH is None:
    raise FileNotFoundError(f"No reference combined inverse+surrogate model found in {MODEL_DIR}")
INVERSE_MODEL_PATH = COMBINED_MODEL_PATH
ACTIVE_SURROGATE_MODEL_PATH = SURROGATE_MODEL_PATH

# Retrain a fresh surrogate for each limited training subset, then freeze it
# while training the inverse model for that same subset.
RETRAIN_SURROGATE_PER_SUBSET = True

# Ten evenly spaced training-pool fractions for the 100-row Ansys validation export.
FRACTIONS = tuple(np.linspace(0.10, 1.00, 10).round(2))
SEEDS = (0,)

REQUESTED_TEST_CORNER_UM = np.array([50.0, 2.0, 420.0])

# I keep test and validation in the same corner region so the validation/test
# losses answer the same question: how well do we learn near the held-out corner?
TEST_FRACTION = 0.15
VAL_FRACTION = 0.15

# Run the training sweep. Set this to False if the CSV already exists and you
# only want to remake the plots.
RUN_SWEEP = True

EPS = 1e-12


In [11]:
# data leading

@dataclass
class Scaler:
    min_: np.ndarray
    max_: np.ndarray

    @property
    def range_(self) -> np.ndarray:
        return np.maximum(self.max_ - self.min_, EPS)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.min_) / self.range_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.range_ + self.min_


def parse_um(value: object) -> float:
    text = str(value).strip()
    for suffix in ("um", "µm", "μm"):
        if text.endswith(suffix):
            return float(text[: -len(suffix)])
    return float(text)


def load_arrays() -> tuple[np.ndarray, np.ndarray]:
    """Load Hamiltonian targets and geometry values from the SQuADDS metadata."""
    data = json.loads(METADATA_PATH.read_text())
    hamiltonian = []
    geometry = []

    for row in data:
        h = row["Hamiltonian_params"]
        opts = row["design"]["design_options"]
        readout = opts["connection_pads"]["readout"]

        hamiltonian.append(
            [
                float(h["qubit_frequency_GHz"]),
                float(h["anharmonicity_MHz"]),
            ]
        )
        geometry.append(
            [
                parse_um(readout["claw_length"]),
                parse_um(readout["ground_spacing"]),
                parse_um(opts["cross_length"]),
            ]
        )

    return np.asarray(hamiltonian, dtype=np.float64), np.asarray(geometry, dtype=np.float64)


def choose_corner_heldout_split(
    geometry_um: np.ndarray,
    requested_corner_um: np.ndarray,
    test_fraction: float = 0.15,
    val_fraction: float = 0.15,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Choose a held-out corner region and keep the remaining points as training candidates.

    The requested corner is clipped to the actual data range because the collaborator's
    approximate values were read by eye and can sit slightly outside the sampled range.
    """
    n_rows = len(geometry_um)
    n_test = int(np.ceil(test_fraction * n_rows))
    n_val = int(np.ceil(val_fraction * n_rows))

    geom_min = geometry_um.min(axis=0)
    geom_max = geometry_um.max(axis=0)
    clipped_corner_um = np.clip(requested_corner_um, geom_min, geom_max)

    corner_scaler = Scaler(geom_min, geom_max)
    geometry_scaled = corner_scaler.transform(geometry_um)
    corner_scaled = corner_scaler.transform(clipped_corner_um.reshape(1, -1))[0]

    corner_distance = np.linalg.norm(geometry_scaled - corner_scaled, axis=1)
    near_corner_order = np.argsort(corner_distance)
    
    corner_idx = near_corner_order[:n_test + n_val]
    rng = np.random.default_rng(42)
    rng.shuffle(corner_idx)
    
    test_idx = corner_idx[:n_test]
    val_idx = corner_idx[n_test:]
    train_pool_idx = near_corner_order[n_test + n_val:]

    return train_pool_idx, val_idx, test_idx, clipped_corner_um, corner_distance


def rank_training_far_to_near(
    geometry_scaled: np.ndarray,
    train_pool_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Rank the training pool by distance to the held-out test corner.

    The first samples are the farthest from the testing data. As the fraction
    increases, the subset grows toward the testing data.
    """
    train_geom = geometry_scaled[train_pool_idx]
    test_geom = geometry_scaled[test_idx]

    nn = NearestNeighbors(n_neighbors=1).fit(test_geom)
    train_to_test_distance = nn.kneighbors(train_geom, return_distance=True)[0][:, 0]
    far_to_near_order = np.argsort(-train_to_test_distance)

    return far_to_near_order, train_to_test_distance


def scaler_from_artifacts(
    values: np.ndarray,
    columns: list[str],
    path_patterns: list[str],
    label: str,
) -> tuple[Scaler, list[str]]:
    """Load per-column MinMaxScaler ranges when present, otherwise fit on metadata."""
    mins = []
    maxs = []
    sources = []

    for i, col in enumerate(columns):
        loaded = None
        source = None
        for pattern in path_patterns:
            candidate = SCALERS_DIR / pattern.format(col=col)
            if candidate.exists():
                loaded = joblib.load(candidate)
                source = str(candidate)
                break

        if loaded is not None:
            mins.append(float(np.asarray(loaded.data_min_).reshape(-1)[0]))
            maxs.append(float(np.asarray(loaded.data_max_).reshape(-1)[0]))
            sources.append(source)
        else:
            mins.append(float(np.min(values[:, i])))
            maxs.append(float(np.max(values[:, i])))
            sources.append(f"metadata fallback: {label}.{col}")

    return Scaler(np.asarray(mins), np.asarray(maxs)), sources


In [12]:
# builing the split and scalars

h_raw, geom_raw_um = load_arrays()
geom_raw_si = geom_raw_um * 1e-6

HAMILTONIAN_COLUMN_NAMES = (METADATA_DIR / "X_names").read_text().splitlines()
QISKIT_PARAM_NAMES = np.load(METADATA_DIR / "y_columns.npy", allow_pickle=True).astype(str).tolist()

train_pool_idx, val_idx, test_idx, clipped_corner_um, corner_distance = choose_corner_heldout_split(
    geom_raw_um,
    REQUESTED_TEST_CORNER_UM,
    TEST_FRACTION,
    VAL_FRACTION,
)

# Make sure the held-out corner data never leaks into the inverse-model training pool.
assert len(np.intersect1d(train_pool_idx, val_idx)) == 0
assert len(np.intersect1d(train_pool_idx, test_idx)) == 0
assert len(np.intersect1d(val_idx, test_idx)) == 0

# The split/ranking scaler is only for choosing far-to-near geometry coverage.
# It is fit on the non-held-out training pool only.
geom_split_scaler = Scaler(
    geom_raw_um[train_pool_idx].min(axis=0),
    geom_raw_um[train_pool_idx].max(axis=0),
)
geom_scaled = geom_split_scaler.transform(geom_raw_um)

far_to_near_order, train_to_test_distance = rank_training_far_to_near(
    geom_scaled,
    train_pool_idx,
    test_idx,
)

# For the actual model inputs/outputs, use the saved scaler artifacts from the
# existing repo workflow. This keeps the already trained surrogate in the same
# scaled space it was originally trained in.
h_model_scaler, h_scaler_sources = scaler_from_artifacts(
    h_raw,
    HAMILTONIAN_COLUMN_NAMES,
    ["scaler_X_{col}.save", "scaler_X_linear_{col}.save"],
    "Hamiltonian",
)
geom_inverse_scaler, geom_inverse_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_{col}_one_hot_encoding.save"],
    "inverse_qiskit",
)
geom_surrogate_scaler, geom_surrogate_scaler_sources = scaler_from_artifacts(
    geom_raw_si,
    QISKIT_PARAM_NAMES,
    ["scaler_y_linear_{col}.save", "scaler_y_{col}_one_hot_encoding.save"],
    "surrogate_qiskit",
)

h_model_scaled = h_model_scaler.transform(h_raw).astype("float32")
geom_inverse_scaled = geom_inverse_scaler.transform(geom_raw_si).astype("float32")
geom_surrogate_scaled = geom_surrogate_scaler.transform(geom_raw_si).astype("float32")

# Convert inverse output scaler space to surrogate input scaler space.
# surrogate_scaled = inverse_scaled * scale_a + scale_b
scale_a = (geom_inverse_scaler.range_ / geom_surrogate_scaler.range_).astype("float32")
scale_b = ((geom_inverse_scaler.min_ - geom_surrogate_scaler.min_) / geom_surrogate_scaler.range_).astype("float32")

print(f"Loaded {len(h_raw)} total samples")
print(f"Training pool: {len(train_pool_idx)}")
print(f"Validation corner: {len(val_idx)}")
print(f"Test corner: {len(test_idx)}")
print()
print("Split check passed:")
print("  train/val overlap:", len(np.intersect1d(train_pool_idx, val_idx)))
print("  train/test overlap:", len(np.intersect1d(train_pool_idx, test_idx)))
print("  val/test overlap:", len(np.intersect1d(val_idx, test_idx)))
print("  100% means all non-held-out training-pool samples, not all samples.")
print()
print("Requested corner [um]:", REQUESTED_TEST_CORNER_UM)
print("Used clipped corner [um]:", clipped_corner_um)
print()
print("Using saved model-space scalers from the existing repo workflow.")
for name, lo, hi, source in zip(HAMILTONIAN_COLUMN_NAMES, h_model_scaler.min_, h_model_scaler.max_, h_scaler_sources):
    print(f"  Hamiltonian {name}: {lo:.6g} to {hi:.6g}  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_inverse_scaler.min_, geom_inverse_scaler.max_, geom_inverse_scaler_sources):
    print(f"  Inverse geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")
for name, lo, hi, source in zip(QISKIT_PARAM_NAMES, geom_surrogate_scaler.min_, geom_surrogate_scaler.max_, geom_surrogate_scaler_sources):
    print(f"  Surrogate geometry {name}: {lo:.6g} to {hi:.6g} SI units  ({source})")

if any(source.startswith("metadata fallback") for source in h_scaler_sources + geom_inverse_scaler_sources + geom_surrogate_scaler_sources):
    print()
    print("Note: at least one scaler artifact was not found, so metadata-derived min/max ranges were used for that column.")


Loaded 1934 total samples
Training pool: 1352
Validation corner: 291
Test corner: 291

Split check passed:
  train/val overlap: 0
  train/test overlap: 0
  val/test overlap: 0
  100% means all non-held-out training-pool samples, not all samples.

Requested corner [um]: [ 50.   2. 420.]
Used clipped corner [um]: [ 70.    4.1 420. ]

Using saved model-space scalers from the existing repo workflow.
  Hamiltonian qubit_frequency_GHz: 3.21853 to 7.12659  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_qubit_frequency_GHz.save)
  Hamiltonian anharmonicity_MHz: -525.818 to -88.9577  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_X_anharmonicity_MHz.save)
  Inverse geometry design_options.connection_pads.readout.claw_length: 7e-05 to 0.0004 SI units  (/home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/scalers/scaler_y_desig

In [13]:

def read_keras_config(path: Path) -> dict:
    with zipfile.ZipFile(path) as zf:
        return json.loads(zf.read("config.json"))


def extract_reference_specs(combined_path: Path) -> tuple[dict, dict]:
    cfg = read_keras_config(combined_path)
    layers = cfg["config"]["layers"]
    inverse_cfg = next(
        layer for layer in layers
        if layer.get("class_name") == "Sequential" and layer.get("config", {}).get("name") == "inverse_model"
    )
    compile_cfg = cfg.get("compile_config") or {}
    return inverse_cfg["config"], compile_cfg


def extract_surrogate_specs(surrogate_path: Path) -> tuple[dict, dict]:
    cfg = read_keras_config(surrogate_path)
    if cfg.get("class_name") != "Sequential":
        raise ValueError(f"Expected a Sequential surrogate model, got {cfg.get('class_name')} from {surrogate_path}")
    return cfg["config"], cfg.get("compile_config") or {}


def _initializer_from_config(config: dict | None, seed: int | None):
    if not config:
        return None
    cfg = json.loads(json.dumps(config))
    if seed is not None and isinstance(cfg.get("config"), dict) and "seed" in cfg["config"]:
        cfg["config"]["seed"] = seed
    try:
        return tf.keras.initializers.deserialize(cfg)
    except Exception:
        class_name = cfg.get("class_name")
        if class_name == "HeNormal":
            return tf.keras.initializers.HeNormal(seed=cfg.get("config", {}).get("seed"))
        if class_name == "LecunUniform":
            return tf.keras.initializers.LecunUniform(seed=cfg.get("config", {}).get("seed"))
        if class_name == "Zeros":
            return tf.keras.initializers.Zeros()
        return tf.keras.initializers.get(class_name)


def _regularizer_from_config(config: dict | None):
    if not config:
        return None
    try:
        return tf.keras.regularizers.deserialize(config)
    except Exception:
        if config.get("class_name") == "L2":
            return tf.keras.regularizers.l2(config.get("config", {}).get("l2", 0.01))
        raise


def build_sequential_from_config(config: dict, input_dim: int, seed: int, default_name: str) -> Sequential:
    tf.keras.utils.set_random_seed(seed)
    model = Sequential(name=config.get("name", default_name))
    for layer_idx, layer_cfg in enumerate(config["layers"]):
        class_name = layer_cfg["class_name"]
        cfg = layer_cfg["config"]
        if class_name == "InputLayer":
            model.add(Input(shape=(input_dim,), name=cfg.get("name", "input")))
        elif class_name == "Dense":
            model.add(
                Dense(
                    cfg["units"],
                    activation=cfg.get("activation", "linear"),
                    name=cfg.get("name"),
                    kernel_initializer=_initializer_from_config(cfg.get("kernel_initializer"), seed + layer_idx),
                    bias_initializer=_initializer_from_config(cfg.get("bias_initializer"), None),
                    kernel_regularizer=_regularizer_from_config(cfg.get("kernel_regularizer")),
                    bias_regularizer=_regularizer_from_config(cfg.get("bias_regularizer")),
                )
            )
        elif class_name == "LeakyReLU":
            model.add(LeakyReLU(negative_slope=cfg.get("negative_slope", 0.01), name=cfg.get("name")))
        elif class_name == "Dropout":
            model.add(Dropout(rate=cfg.get("rate", 0.0), name=cfg.get("name")))
        else:
            raise ValueError(f"Unsupported reference layer type: {class_name}")
    return model


def build_inverse_from_reference(input_dim: int, seed: int) -> Sequential:
    return build_sequential_from_config(REFERENCE_INVERSE_CONFIG, input_dim, seed, "inverse_model")


def build_surrogate_from_reference(input_dim: int, seed: int) -> Sequential:
    model = build_sequential_from_config(SURROGATE_REFERENCE_CONFIG, input_dim, seed, "retrained_surrogate")
    model.compile(
        optimizer=build_surrogate_optimizer(),
        loss=SURROGATE_RECONSTRUCTION_LOSS,
        metrics=[SURROGATE_TRAIN_LOSS],
        jit_compile=SURROGATE_JIT_COMPILE,
    )
    return model


class ScalerConversionLayer(tf.keras.layers.Layer):
    def __init__(self, scale_a, scale_b, **kwargs):
        kwargs.setdefault("trainable", False)
        super().__init__(**kwargs)
        self._scale_a = tf.constant(scale_a, dtype=tf.float32)
        self._scale_b = tf.constant(scale_b, dtype=tf.float32)
        self._cfg = {"scale_a": list(np.asarray(scale_a, dtype=float)), "scale_b": list(np.asarray(scale_b, dtype=float))}

    def call(self, inputs):
        a = tf.cast(self._scale_a, inputs.dtype)
        b = tf.cast(self._scale_b, inputs.dtype)
        return inputs * a + b

    def get_config(self):
        config = super().get_config()
        config.update(self._cfg)
        return config


def qiskit_range_penalty(y_true_dummy, y_pred):
    """Penalize inverse predictions outside the scaled [0, 1] training range."""
    below = tf.nn.relu(-y_pred)
    above = tf.nn.relu(y_pred - 1.0)
    return tf.reduce_mean(below ** 2 + above ** 2)


def load_frozen_surrogate(surrogate_model_path: Path):
    surrogate_model = load_model(surrogate_model_path, compile=False)
    surrogate_model.trainable = False
    for layer in surrogate_model.layers:
        layer.trainable = False
    return surrogate_model


def deserialize_reference_learning_rate(lr_config):
    if isinstance(lr_config, (int, float, np.integer, np.floating)):
        return float(lr_config)
    if isinstance(lr_config, str):
        try:
            return float(lr_config)
        except ValueError:
            return lr_config
    if isinstance(lr_config, dict):
        cfg = lr_config.get("config", {})
        for key in ("value", "initial_value"):
            if key in cfg and isinstance(cfg[key], (int, float, np.integer, np.floating)):
                return float(cfg[key])
        try:
            return tf.keras.optimizers.schedules.deserialize(lr_config)
        except Exception:
            if "initial_learning_rate" in cfg:
                return tf.keras.optimizers.schedules.ExponentialDecay(
                    initial_learning_rate=float(cfg["initial_learning_rate"]),
                    decay_steps=cfg.get("decay_steps", 1),
                    decay_rate=cfg.get("decay_rate", 1.0),
                    staircase=cfg.get("staircase", False),
                )
            raise
    return lr_config


def describe_learning_rate(lr) -> str:
    if isinstance(lr, (int, float, np.integer, np.floating)):
        return str(float(lr))
    if isinstance(lr, tf.keras.optimizers.schedules.LearningRateSchedule):
        return f"{lr.__class__.__name__}({lr.get_config()})"
    return str(lr)


def optimizer_lr_label(compile_config: dict) -> str:
    try:
        lr_cfg = compile_config["optimizer"]["config"]["learning_rate"]
        return describe_learning_rate(deserialize_reference_learning_rate(lr_cfg))
    except Exception:
        return "unknown"


def build_reference_optimizer():
    try:
        return tf.keras.optimizers.deserialize(REFERENCE_COMPILE_CONFIG["optimizer"])
    except Exception:
        return tf.keras.optimizers.Adam(learning_rate=REFERENCE_LEARNING_RATE)


def build_surrogate_optimizer():
    try:
        return tf.keras.optimizers.deserialize(SURROGATE_COMPILE_CONFIG["optimizer"])
    except Exception:
        return tf.keras.optimizers.Adam()


def build_combined_model(seed: int, surrogate_model_path: Path) -> tuple[Sequential, Model]:
    tf.keras.backend.clear_session()
    surrogate_model = load_frozen_surrogate(surrogate_model_path)
    inverse_model = build_inverse_from_reference(h_model_scaled.shape[1], seed=seed)

    combined_input = Input(shape=(h_model_scaled.shape[1],), name="combined_input")
    predicted_qiskit = inverse_model(combined_input)
    predicted_qiskit_converted = ScalerConversionLayer(scale_a, scale_b, name="scaler_conversion")(predicted_qiskit)
    reconstructed_hamiltonian = surrogate_model(predicted_qiskit_converted)

    combined_model = Model(
        inputs=combined_input,
        outputs=[reconstructed_hamiltonian, predicted_qiskit],
        name="combined_model",
    )

    combined_model.compile(
        optimizer=build_reference_optimizer(),
        loss=[REFERENCE_RECONSTRUCTION_LOSS, qiskit_range_penalty],
        loss_weights=REFERENCE_LOSS_WEIGHTS,
        jit_compile=REFERENCE_JIT_COMPILE,
    )

    return inverse_model, combined_model


REFERENCE_INVERSE_CONFIG, REFERENCE_COMPILE_CONFIG = extract_reference_specs(COMBINED_MODEL_PATH)
SURROGATE_REFERENCE_CONFIG, SURROGATE_COMPILE_CONFIG = extract_surrogate_specs(SURROGATE_MODEL_PATH)

REFERENCE_OPTIMIZER_CONFIG = REFERENCE_COMPILE_CONFIG["optimizer"]["config"]
REFERENCE_LEARNING_RATE = deserialize_reference_learning_rate(REFERENCE_OPTIMIZER_CONFIG["learning_rate"])
REFERENCE_LEARNING_RATE_LABEL = describe_learning_rate(REFERENCE_LEARNING_RATE)
REFERENCE_RECONSTRUCTION_LOSS = REFERENCE_COMPILE_CONFIG["loss"][0]
REFERENCE_LOSS_WEIGHTS = [float(v) for v in REFERENCE_COMPILE_CONFIG.get("loss_weights", [1.0, 1.0])]
# Pin the out-of-range (qiskit_range_penalty) weight to the best optimized value
# from the ml_21 Keras-tuner search. The best trial (val loss 0.001002) used
# penalty_weight = 1.0, matching the saved best combined model loss_weights.
# Setting it explicitly keeps this sweep identical to the best model regardless
# of which reference model file is loaded.
BEST_PENALTY_WEIGHT = 1.0
REFERENCE_LOSS_WEIGHTS = [REFERENCE_LOSS_WEIGHTS[0], BEST_PENALTY_WEIGHT]
REFERENCE_JIT_COMPILE = bool(REFERENCE_COMPILE_CONFIG.get("jit_compile", False))
REFERENCE_BATCH_SIZE = TRAIN_BATCH_SIZE
REFERENCE_EPOCHS = EPOCHS
REFERENCE_EARLY_STOPPING_PATIENCE = TRAIN_EARLY_STOPPING_PATIENCE

SURROGATE_RECONSTRUCTION_LOSS = SURROGATE_COMPILE_CONFIG.get("loss", SURROGATE_TRAIN_LOSS)
SURROGATE_LEARNING_RATE_LABEL = optimizer_lr_label(SURROGATE_COMPILE_CONFIG)
SURROGATE_JIT_COMPILE = bool(SURROGATE_COMPILE_CONFIG.get("jit_compile", False))

reference_dense_layers = [
    layer["config"]["units"]
    for layer in REFERENCE_INVERSE_CONFIG["layers"]
    if layer["class_name"] == "Dense"
]
surrogate_dense_layers = [
    layer["config"]["units"]
    for layer in SURROGATE_REFERENCE_CONFIG["layers"]
    if layer["class_name"] == "Dense"
]
print("Reference inverse dense units:", reference_dense_layers)
print("Reference inverse learning rate:", REFERENCE_LEARNING_RATE_LABEL)
print("Reference inverse reconstruction loss:", REFERENCE_RECONSTRUCTION_LOSS)
print("Reference inverse loss weights:", REFERENCE_LOSS_WEIGHTS)
print("Reference inverse jit_compile:", REFERENCE_JIT_COMPILE)
print("Inverse epochs/batch/patience:", REFERENCE_EPOCHS, REFERENCE_BATCH_SIZE, REFERENCE_EARLY_STOPPING_PATIENCE)
print()
print("Reference surrogate dense units:", surrogate_dense_layers)
print("Reference surrogate learning rate:", SURROGATE_LEARNING_RATE_LABEL)
print("Reference surrogate loss:", SURROGATE_RECONSTRUCTION_LOSS)
print("Reference surrogate jit_compile:", SURROGATE_JIT_COMPILE)
print("Surrogate epochs/batch/patience:", SURROGATE_EPOCHS, SURROGATE_TRAIN_BATCH_SIZE, SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE)


Reference inverse dense units: [64, 3]
Reference inverse learning rate: ExponentialDecay({'initial_learning_rate': 0.001, 'decay_steps': 220, 'decay_rate': 0.99, 'staircase': False, 'name': 'ExponentialDecay'})
Reference inverse reconstruction loss: mae
Reference inverse loss weights: [1.0, 1.0]
Reference inverse jit_compile: True
Inverse epochs/batch/patience: 400 128 60

Reference surrogate dense units: [736, 2]
Reference surrogate learning rate: 0.05013880506157875
Reference surrogate loss: mae
Reference surrogate jit_compile: True
Surrogate epochs/batch/patience: 400 128 60


In [14]:
def fit_surrogate_for_subset(
    geom_subset_scaled: np.ndarray,
    h_subset_scaled: np.ndarray,
    geom_val_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    seed: int,
) -> tuple[Sequential, tf.keras.callbacks.History]:
    surrogate_model = build_surrogate_from_reference(geom_subset_scaled.shape[1], seed=seed)
    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )
    history = surrogate_model.fit(
        np.asarray(geom_subset_scaled, dtype="float32"),
        np.asarray(h_subset_scaled, dtype="float32"),
        epochs=SURROGATE_EPOCHS,
        batch_size=SURROGATE_TRAIN_BATCH_SIZE,
        validation_data=(
            np.asarray(geom_val_scaled, dtype="float32"),
            np.asarray(h_val_scaled, dtype="float32"),
        ),
        callbacks=[early_stopping],
        verbose=0,
    )
    return surrogate_model, history


def fit_inverse_for_subset(
    h_subset_scaled: np.ndarray,
    h_val_scaled: np.ndarray,
    seed: int,
    surrogate_model_path: Path,
) -> tuple[Sequential, Model, tf.keras.callbacks.History]:
    inverse_model, combined_model = build_combined_model(seed=seed, surrogate_model_path=surrogate_model_path)
    dummy_train = np.zeros((len(h_subset_scaled), len(QISKIT_PARAM_NAMES)), dtype="float32")
    dummy_val = np.zeros((len(h_val_scaled), len(QISKIT_PARAM_NAMES)), dtype="float32")

    early_stopping = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=REFERENCE_EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=0,
    )

    history = combined_model.fit(
        np.asarray(h_subset_scaled, dtype="float32"),
        [np.asarray(h_subset_scaled, dtype="float32"), dummy_train],
        epochs=REFERENCE_EPOCHS,
        batch_size=REFERENCE_BATCH_SIZE,
        validation_data=(
            np.asarray(h_val_scaled, dtype="float32"),
            [np.asarray(h_val_scaled, dtype="float32"), dummy_val],
        ),
        callbacks=[early_stopping],
        verbose=0,
    )

    return inverse_model, combined_model, history


def evaluate_percent_error(
    combined_model: Model,
    h_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict[str, float]:
    h_pred_scaled, _ = combined_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)

    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def evaluate_surrogate_model(
    surrogate_model: Model,
    geom_scaled_in: np.ndarray,
    h_unscaled: np.ndarray,
) -> dict[str, float]:
    h_pred_scaled = surrogate_model.predict(np.asarray(geom_scaled_in, dtype="float32"), verbose=0)
    h_pred = h_model_scaler.inverse_transform(h_pred_scaled)
    pct = 100.0 * np.abs(h_pred - h_unscaled) / np.maximum(np.abs(h_unscaled), EPS)
    return {
        "omega_q_mean_pct": float(np.mean(pct[:, 0])),
        "alpha_mean_pct": float(np.mean(pct[:, 1])),
        "mean_hamiltonian_pct": float(np.mean(pct)),
    }


def inverse_range_stats(inverse_model: Sequential, h_scaled_in: np.ndarray) -> dict[str, float]:
    qiskit_scaled = inverse_model.predict(np.asarray(h_scaled_in, dtype="float32"), verbose=0)
    below = np.maximum(-qiskit_scaled, 0.0)
    above = np.maximum(qiskit_scaled - 1.0, 0.0)
    violation = below + above
    return {
        "qiskit_scaled_min": float(np.min(qiskit_scaled)),
        "qiskit_scaled_max": float(np.max(qiskit_scaled)),
        "qiskit_range_violation_mean": float(np.mean(violation)),
        "qiskit_range_violation_max": float(np.max(violation)),
    }


In [15]:
# surrogate training mode

print("Retraining a fresh surrogate for each fraction/seed using the same subset as the inverse model.")
print("Surrogate reference model:", SURROGATE_MODEL_PATH)
print("Sweep model output dir:", SWEEP_MODEL_DIR)


Retraining a fresh surrogate for each fraction/seed using the same subset as the inverse model.
Surrogate reference model: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/best_keras_model_model2_surrogate.keras
Sweep model output dir: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/corner_far_to_near_data_amount_sweep_retrained_surrogate


In [16]:
# run sweep
def model_paths_for_fraction_seed(fraction: float, seed: int) -> tuple[Path, Path, Path]:
    pct = int(round(fraction * 100))
    stem = f"fraction_{pct:03d}pct_seed{seed}"
    return (
        SWEEP_MODEL_DIR / f"{stem}_surrogate.keras",
        SWEEP_MODEL_DIR / f"{stem}_combined.keras",
        SWEEP_MODEL_DIR / f"{stem}_inverse.keras",
    )


subset_indices_by_fraction = {}

if RUN_SWEEP:
    print(f"Retraining surrogate for each subset from reference: {SURROGATE_MODEL_PATH}")
    print(f"Copying inverse architecture/compile hyperparameters from: {COMBINED_MODEL_PATH}")

    rows = []
    n_train_pool = len(train_pool_idx)
    hidden_units = [
        layer["config"]["units"]
        for layer in REFERENCE_INVERSE_CONFIG["layers"]
        if layer["class_name"] == "Dense"
    ]
    surrogate_hidden_units = [
        layer["config"]["units"]
        for layer in SURROGATE_REFERENCE_CONFIG["layers"]
        if layer["class_name"] == "Dense"
    ]

    for fraction in FRACTIONS:
        n_subset = max(1, int(round(fraction * n_train_pool)))
        subset_local = far_to_near_order[:n_subset]
        subset_idx = train_pool_idx[subset_local]
        subset_dist = train_to_test_distance[subset_local]
        subset_indices_by_fraction[fraction] = subset_idx

        print(f"Training surrogate+inverse using {fraction:.0%} of training pool ({n_subset} samples)...")

        for seed in SEEDS:
            surrogate_model_path, combined_model_path, inverse_model_path = model_paths_for_fraction_seed(fraction, seed)

            surrogate_model, surrogate_history = fit_surrogate_for_subset(
                geom_surrogate_scaled[subset_idx],
                h_model_scaled[subset_idx],
                geom_surrogate_scaled[val_idx],
                h_model_scaled[val_idx],
                seed=seed,
            )
            surrogate_model.save(surrogate_model_path)

            surrogate_train_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[subset_idx],
                h_raw[subset_idx],
            )
            surrogate_val_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[val_idx],
                h_raw[val_idx],
            )
            surrogate_test_metrics = evaluate_surrogate_model(
                surrogate_model,
                geom_surrogate_scaled[test_idx],
                h_raw[test_idx],
            )

            del surrogate_model
            tf.keras.backend.clear_session()
            gc.collect()

            inverse_model, combined_model, history = fit_inverse_for_subset(
                h_model_scaled[subset_idx],
                h_model_scaled[val_idx],
                seed=seed,
                surrogate_model_path=surrogate_model_path,
            )
            combined_model.save(combined_model_path)
            inverse_model.save(inverse_model_path)

            train_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[subset_idx],
                h_raw[subset_idx],
            )
            val_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[val_idx],
                h_raw[val_idx],
            )
            test_metrics = evaluate_percent_error(
                combined_model,
                h_model_scaled[test_idx],
                h_raw[test_idx],
            )
            test_range_stats = inverse_range_stats(inverse_model, h_model_scaled[test_idx])

            rows.append(
                {
                    "selection_method": "corner_test_far_to_near_by_test_geometry_nn_distance_retrained_surrogate",
                    "requested_corner_claw_um": REQUESTED_TEST_CORNER_UM[0],
                    "requested_corner_ground_um": REQUESTED_TEST_CORNER_UM[1],
                    "requested_corner_cross_um": REQUESTED_TEST_CORNER_UM[2],
                    "used_corner_claw_um": clipped_corner_um[0],
                    "used_corner_ground_um": clipped_corner_um[1],
                    "used_corner_cross_um": clipped_corner_um[2],
                    "fraction": fraction,
                    "training_percent": fraction * 100.0,
                    "n_samples": n_subset,
                    "seed": seed,
                    "surrogate_reference_model_path": str(SURROGATE_MODEL_PATH),
                    "surrogate_training_mode": "retrained_on_same_subset_as_inverse",
                    "combined_reference_model_path": str(COMBINED_MODEL_PATH),
                    "inverse_reference_model_path": str(INVERSE_MODEL_PATH),
                    "saved_surrogate_model_path": str(surrogate_model_path),
                    "saved_combined_model_path": str(combined_model_path),
                    "saved_inverse_model_path": str(inverse_model_path),
                    "surrogate_dense_units": json.dumps(surrogate_hidden_units),
                    "inverse_dense_units": json.dumps(hidden_units),
                    "surrogate_optimizer": "Adam",
                    "surrogate_learning_rate": SURROGATE_LEARNING_RATE_LABEL,
                    "surrogate_reconstruction_loss": SURROGATE_RECONSTRUCTION_LOSS,
                    "surrogate_jit_compile": SURROGATE_JIT_COMPILE,
                    "surrogate_epochs_run": len(surrogate_history.history.get("loss", [])),
                    "surrogate_batch_size": SURROGATE_TRAIN_BATCH_SIZE,
                    "surrogate_early_stopping_patience": SURROGATE_TRAIN_EARLY_STOPPING_PATIENCE,
                    "inverse_optimizer": "Adam",
                    "inverse_learning_rate": REFERENCE_LEARNING_RATE_LABEL,
                    "inverse_reconstruction_loss": REFERENCE_RECONSTRUCTION_LOSS,
                    "range_penalty_weight": REFERENCE_LOSS_WEIGHTS[1],
                    "inverse_jit_compile": REFERENCE_JIT_COMPILE,
                    "inverse_epochs_run": len(history.history.get("loss", [])),
                    "inverse_batch_size": REFERENCE_BATCH_SIZE,
                    "inverse_early_stopping_patience": REFERENCE_EARLY_STOPPING_PATIENCE,
                    "subset_to_test_nn_distance_min": float(np.min(subset_dist)),
                    "subset_to_test_nn_distance_median": float(np.median(subset_dist)),
                    "subset_to_test_nn_distance_max": float(np.max(subset_dist)),
                    **{f"surrogate_train_{key}": value for key, value in surrogate_train_metrics.items()},
                    **{f"surrogate_val_{key}": value for key, value in surrogate_val_metrics.items()},
                    **{f"surrogate_test_{key}": value for key, value in surrogate_test_metrics.items()},
                    **{f"train_{key}": value for key, value in train_metrics.items()},
                    **{f"val_{key}": value for key, value in val_metrics.items()},
                    **{f"test_{key}": value for key, value in test_metrics.items()},
                    **{f"test_{key}": value for key, value in test_range_stats.items()},
                }
            )

            del inverse_model, combined_model, history, surrogate_history
            tf.keras.backend.clear_session()
            gc.collect()

    out = pd.DataFrame(rows)
    out.to_csv(OUT_PATH, index=False)
    print()
    print(f"wrote {OUT_PATH}")
    print(f"saved sweep models to {SWEEP_MODEL_DIR}")
else:
    print(f"Skipping training. Reading existing results from {OUT_PATH}")
    out = pd.read_csv(OUT_PATH)

out.head()


Retraining surrogate for each subset from reference: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/best_keras_model_model2_surrogate.keras
Copying inverse architecture/compile hyperparameters from: /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/model/surrogate_loss_2in_3out_best_model.keras
Training surrogate+inverse using 10% of training pool (135 samples)...
Training surrogate+inverse using 20% of training pool (270 samples)...
Training surrogate+inverse using 30% of training pool (406 samples)...
Training surrogate+inverse using 40% of training pool (541 samples)...
Training surrogate+inverse using 50% of training pool (676 samples)...
Training surrogate+inverse using 60% of training pool (811 samples)...
Training surrogate+inverse using 70% of training pool (946 samples)...
Training surrogate+inverse using 80% of training pool (1082 samples)...
Training surrogate+inverse using 90%

,selection_method,requested_corner_claw_um,requested_corner_ground_um,requested_corner_cross_um,used_corner_claw_um,used_corner_ground_um,used_corner_cross_um,fraction,training_percent,n_samples,...,val_omega_q_mean_pct,val_alpha_mean_pct,val_mean_hamiltonian_pct,test_omega_q_mean_pct,test_alpha_mean_pct,test_mean_hamiltonian_pct,test_qiskit_scaled_min,test_qiskit_scaled_max,test_qiskit_range_violation_mean,test_qiskit_range_violation_max
0,corner_test_far_to_near_by_test_geometry_nn_di...,50.0,2.0,420.0,70.0,4.1,420.0,0.1,10.0,135,...,6.814485,3.429754,5.122120,6.106243,3.509131,4.807687,0.018352,1.140146,0.021448,0.140146
1,corner_test_far_to_near_by_test_geometry_nn_di...,50.0,2.0,420.0,70.0,4.1,420.0,0.2,20.0,270,...,0.465299,0.937732,0.701515,0.460487,0.846027,0.653257,0.307857,1.159264,0.012993,0.159264
2,corner_test_far_to_near_by_test_geometry_nn_di...,50.0,2.0,420.0,70.0,4.1,420.0,0.3,30.0,406,...,0.284978,0.297228,0.291103,0.207033,0.299934,0.253484,-0.041333,0.514930,0.011358,0.041333
3,corner_test_far_to_near_by_test_geometry_nn_di...,50.0,2.0,420.0,70.0,4.1,420.0,0.4,40.0,541,...,0.123501,0.249611,0.186556,0.115862,0.251543,0.183703,-0.010791,1.055815,0.003350,0.055815
4,corner_test_far_to_near_by_test_geometry_nn_di...,50.0,2.0,420.0,70.0,4.1,420.0,0.5,50.0,676,...,0.150517,0.641374,0.395946,0.149507,0.565726,0.357617,-0.546319,0.703484,0.146851,0.546319


In [17]:
summary = (
    out.groupby(["training_percent", "n_samples"], as_index=False)
    .agg(
        train_mean=("train_mean_hamiltonian_pct", "mean"),
        train_std=("train_mean_hamiltonian_pct", "std"),
        val_mean=("val_mean_hamiltonian_pct", "mean"),
        val_std=("val_mean_hamiltonian_pct", "std"),
        test_mean=("test_mean_hamiltonian_pct", "mean"),
        test_std=("test_mean_hamiltonian_pct", "std"),
        dist_min=("subset_to_test_nn_distance_min", "mean"),
        dist_median=("subset_to_test_nn_distance_median", "mean"),
        dist_max=("subset_to_test_nn_distance_max", "mean"),
    )
    .sort_values("training_percent")
)
summary.to_csv(SUMMARY_OUT_PATH, index=False)
print(f"wrote {SUMMARY_OUT_PATH}")
summary


wrote /home/olivias/ML_qubit_design/experiments/model_predict_qubit_TransmonCross_Hamiltonian_params/data_amount_sweep_corner_far_to_near_retrained_surrogate_summary.csv


,training_percent,n_samples,train_mean,train_std,val_mean,val_std,test_mean,test_std,dist_min,dist_median,dist_max
0,10.0,135,7.368846,NaN,5.122120,NaN,4.807687,NaN,0.865682,0.875701,0.971165
1,20.0,270,2.197173,NaN,0.701515,NaN,0.653257,NaN,0.852320,0.865682,0.971165
2,30.0,406,2.119731,NaN,0.291103,NaN,0.253484,NaN,0.847999,0.856618,0.971165
3,40.0,541,0.255079,NaN,0.186556,NaN,0.183703,NaN,0.847458,0.852320,0.971165
4,50.0,676,0.418054,NaN,0.395946,NaN,0.357617,NaN,0.701491,0.849622,0.971165
5,60.0,811,0.476537,NaN,0.427313,NaN,0.445238,NaN,0.695576,0.847999,0.971165
6,70.0,946,0.662440,NaN,0.678140,NaN,0.671136,NaN,0.694915,0.847999,0.971165
7,80.0,1082,0.252024,NaN,0.109378,NaN,0.107926,NaN,0.181818,0.847458,0.971165
8,90.0,1217,0.121868,NaN,0.084950,NaN,0.081080,NaN,0.085710,0.741580,0.971165
9,100.0,1352,0.179201,NaN,0.191186,NaN,0.177148,NaN,0.030303,0.701491,0.971165


In [18]:
EXPORT_DIR = EXPERIMENT_DIR / "results" / "validation" / "data_amount_sweep_heldout_corner_ansys"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_FRACTIONS = tuple(np.linspace(0.10, 1.00, 10).round(2))
EXPORT_SEEDS = (0,)
N_EXPORT_TEST_SAMPLES = 10

EXPORT_PATH = EXPORT_DIR / "heldout_corner_test_ansys_input_all_fractions.csv"

eval_idx = np.asarray(test_idx[:N_EXPORT_TEST_SAMPLES], dtype=int)
assert len(np.intersect1d(eval_idx, train_pool_idx)) == 0
assert len(np.intersect1d(eval_idx, val_idx)) == 0

custom_objects = {
    "ScalerConversionLayer": ScalerConversionLayer,
    "qiskit_range_penalty": qiskit_range_penalty,
}

missing = []
for fraction in EXPORT_FRACTIONS:
    for seed in EXPORT_SEEDS:
        _, combined_path, _ = model_paths_for_fraction_seed(float(fraction), int(seed))
        if not combined_path.exists():
            missing.append(str(combined_path))

if missing:
    raise FileNotFoundError(
        "Missing saved models. Run the ml_32 training cell for these fractions first:\n"
        + "\n".join(missing)
    )

rows = []

for fraction in EXPORT_FRACTIONS:
    pct_int = int(round(float(fraction) * 100))

    for seed in EXPORT_SEEDS:
        surrogate_path, combined_path, inverse_path = model_paths_for_fraction_seed(float(fraction), int(seed))
        combined_model = load_model(combined_path, compile=False, custom_objects=custom_objects)

        pred_h_scaled, pred_geom_inverse_scaled = combined_model.predict(
            h_model_scaled[eval_idx].astype("float32"),
            verbose=0,
        )

        pred_h_raw = h_model_scaler.inverse_transform(pred_h_scaled)
        pred_geom_si = geom_inverse_scaler.inverse_transform(pred_geom_inverse_scaled)
        pred_geom_um = pred_geom_si * 1e6

        target_h_raw = h_raw[eval_idx]
        surrogate_pct = 100.0 * np.abs(pred_h_raw - target_h_raw) / np.maximum(np.abs(target_h_raw), EPS)

        for sample_no, idx in enumerate(eval_idx):
            rows.append(
                {
                    "training_percent": float(fraction) * 100.0,
                    "seed": int(seed),
                    "test_sample_number": int(sample_no),
                    
                    "ref_qubit_frequency_GHz": h_raw[idx, 0],
                    "ref_anharmonicity_MHz": h_raw[idx, 1],
                    "ref_connection_pads.readout.claw_length": geom_raw_si[idx, 0],
                    "ref_connection_pads.readout.ground_spacing": geom_raw_si[idx, 1],
                    "ref_cross_length": geom_raw_si[idx, 2],
                    "ref_connection_pads.readout.claw_length_um": geom_raw_um[idx, 0],
                    "ref_connection_pads.readout.ground_spacing_um": geom_raw_um[idx, 1],
                    "ref_cross_length_um": geom_raw_um[idx, 2],

                    "pred_qubit_frequency_GHz": pred_h_raw[sample_no, 0],
                    "pred_anharmonicity_MHz": pred_h_raw[sample_no, 1],
                    "pred_connection_pads.readout.claw_length": pred_geom_si[sample_no, 0],
                    "pred_connection_pads.readout.ground_spacing": pred_geom_si[sample_no, 1],
                    "pred_cross_length": pred_geom_si[sample_no, 2],
                    "pred_connection_pads.readout.claw_length_um": pred_geom_um[sample_no, 0],
                    "pred_connection_pads.readout.ground_spacing_um": pred_geom_um[sample_no, 1],
                    "pred_cross_length_um": pred_geom_um[sample_no, 2],

                    "surrogate_frequency_pct_error": surrogate_pct[sample_no, 0],
                    "surrogate_anharmonicity_pct_error": surrogate_pct[sample_no, 1],
                    "surrogate_mean_hamiltonian_pct_error": surrogate_pct[sample_no].mean(),

                    "ansys_qubit_frequency_GHz": np.nan,
                    "ansys_anharmonicity_MHz": np.nan,
                    "ansys_frequency_pct_error": np.nan,
                    "ansys_anharmonicity_pct_error": np.nan,

                    "saved_surrogate_model_path": str(surrogate_path),
                    "saved_combined_model_path": str(combined_path),
                    "saved_inverse_model_path": str(inverse_path),
                }
            )

        del combined_model
        tf.keras.backend.clear_session()
        gc.collect()

ansys_export_columns = [

    "training_percent",
    "seed",
    "test_sample_number",

    "ref_qubit_frequency_GHz",
    "ref_anharmonicity_MHz",
    "ref_connection_pads.readout.claw_length",
    "ref_connection_pads.readout.ground_spacing",
    "ref_cross_length",

    "pred_qubit_frequency_GHz",
    "pred_anharmonicity_MHz",
    "pred_connection_pads.readout.claw_length",
    "pred_connection_pads.readout.ground_spacing",
    "pred_cross_length",

    "ansys_qubit_frequency_GHz",
    "ansys_anharmonicity_MHz",
    "ansys_frequency_pct_error",
    "ansys_anharmonicity_pct_error",
]

export_df = export_df[ansys_export_columns]

export_df.to_csv(EXPORT_PATH, index=False)

print(f"Wrote {len(export_df)} rows to:")
print(EXPORT_PATH)
export_df.head()

NameError: name 'export_df' is not defined